# MCMRSimulator workshop: the basics

Welcome to the MCMRSimulator practical. This notebook introduces the
simulator and its building blocks, then demonstrates how to combine them
to simulate diffusion-weighted MRI signals. We will connect three pieces
of a simulation:

1.  a geometry, which describes how spins move through tissue;
2.  an MRI sequence, which describes the applied RF pulses, gradient
    waveforms, and readouts;
3.  biophysical parameters, which determine diffusion and relaxation. In
    later sections we will also add magnetisation transfer and magnetic
    susceptibility.

The examples use micrometres for distances and milliseconds for time.
Start by importing the simulator (`MCMRSimulator`), sequence builder
(`MRIBuilder`), and plotting library (`CairoMakie`):

In [1]:
using Random
using MCMRSimulator
using MRIBuilder
using CairoMakie

Random.seed!(1)
set_theme!(Theme(fontsize=20))

## 1. Choose a geometry

We begin with parallel cylinders as a simple axon-like geometry. Two
cylinders repeat in a rectangular unit cell, offset so the full periodic
arrangement is hexagonal. Their centres are `2 μm` apart, and their
radius of `0.9 μm` leaves a small gap between neighbouring cylinders.

In [1]:
spacing = 2.0  # nearest centre-to-centre distance, μm
geometry = Cylinders(
    radius=0.9,
    position=[[0.0, 0.0], [spacing / 2, sqrt(3) * spacing / 2]],
    repeats=(spacing, sqrt(3) * spacing),
)

geometry_figure = plot(PlotPlane(size=8), geometry)
# Keep equal x/y scales so the cylindrical cross-sections appear circular.
geometry_figure.axis.aspect = DataAspect()
geometry_figure

Note the use of a
[`PlotPlane`](https://open.oxcin.ox.ac.uk/pages/ndcn0236/mcmrsimulator.jl/stable/api/#MCMRSimulator.Plot.PlotPlane)
to select the slice of the geometry to plot in the 3D geometry. We can
also call `plot` without a `PlotPlane` to see the full 3D geometry,
although the cylinders will have to be cut off due to the finite
plotting volume:

In [1]:
plot(geometry)

The geometry constrains the movement of spins, which will affect the
measured signal. Without any explicit permeability, spins cannot cross
the cylinder walls, leading to completely separate intra-cylinder and
extra-cylinder compartments. The simulator can also model permeable
membranes, which we will explore in a later section.

Try changing `radius` or `spacing` and rerun this cell. Note that when
the radius becomes too large, the cylinders will overlap. The resulting
geometry will still run in the simulator, but the physical
interpretation is unclear as the cylinders are now split into smaller
compartments. The simulator does not check for overlaps, so it is up to
the user to ensure that the geometry is physically meaningful.

For more geometry options, see the [MCMRSimulator Geometry
documentation](https://pages.fmrib.ox.ac.uk/ndcn0236/mcmrsimulator.jl/dev/geometry/#geometry).

The simulator includes several geometry types for representing tissue:

- **Walls** model parallel planes
- **Cylinders** model infinitely long fibres; **Annuli** can represent
  fibres with a surrounding myelin sheath.
- **Spheres** represent spherical cells; overlapping spheres can be used
  to model more complex cell morphologies as used in the [CATERPillar
  phantom generator](https://github.com/jazz031195/CATERPillar_preview).
- **FiniteCylinders** represent connected morphologies, such as trees
  loaded from [SWC
  files](https://swc-specification.readthedocs.io/en/latest/swc.html).
  These can be used to model cell morphologies downloaded from
  [NeuroMorpho.Org](https://neuromorpho.org/) or generated by the
  [Virtual Cell
  generator](https://github.com/palombom/Virtual-Cell-Generator-v1.0).
- **BendyCylinders** represent curved or beading fibres, such as axons
  in the brain.
- **Meshes** describe arbitrary cell morphologies. While most general,
  representing cell morphologies using `FiniteCylinders` or `Spheres`
  can be more efficient for simulation.
- **Liminal geometries** represent statistical populations of cells
  without explicitly packing all cells into one global arrangement. We
  will see these in action in the next workshop section.

## 2. Choose a sequence

Next we create a diffusion-weighted spin-echo sequence with a b-value of
3. The diffusion gradients point along +x, perpendicular to the cylinder
axes, which are along z.
[`MRIBuilder`](https://open.oxcin.ox.ac.uk/pages/ndcn0236/mribuilder.jl/stable/)
takes gradient orientation as a vector, so `[1, 0, 0]` explicitly
selects the x direction.

In [1]:
base_sequence = MRIBuilder.DiffusionSpinEcho(
    bval=3,
    gradient=(orientation=[1.0, 0.0, 0.0],),
)
MRIBuilder.plot_sequence(base_sequence)


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************


<img width=600 height=450 style='object-fit: contain; height: auto;' src="data:image/png;base64, iVBORw0KGgoAAAANSUhEUgAAAlgAAAHCCAYAAAAzc7dkAAAAAXNSR0IArs4c6QAAAARnQU1BAACxjwv8YQUAAAAgY0hSTQAAeiYAAICEAAD6AAAAgOgAAHUwAADqYAAAOpgAABdwnLpRPAAAAAlwSFlzAAAOxAAADsQBlSsOGwAAIABJREFUeAHtwQ2U1nPC//HPzPW5VDORJCPTqNRINUQtSXmshBuLVWGjUptsaK26t1arrPUUlvWQJSulJpK9cVvsijtJuVtPKetoKkVLredaddX1nfr/f+f853/mdCrhd+V31fv18ub/SwAAAIiNBQAAgFhZAAAAiJUFAACAWFkAAACIlQUAAIBYWQAAAIiVBQAAgFhZAAAAiJUFAACAWFkAAACIlQUAAIBYWQAAAIiVBQAAgFhZAAAAiJUFAACAWFkAAACIlQUAAIBYWQAAAIiVBQAAgFhZAAAAiJUFAACAWFkAAACIlQUAAIBYWQAAAIiVBQAAgFhZAAAAiJUFAACAWFkAAACIlQUAAIBYWQAAAIiVBQAAgFhZALAbu+WWW7Rs2TLVaNu2rS6//HIBwPdhAcBu7M9//rNeffVV1Tj55JN1+eWXCwC+DwsAAACxsgAAABArCwAAALGyAAAAECsLAAAAsbIAAAAQKwsAAACxsgAAABArCwAAALGyAAAAECsLAAAAsbIAAAAQKwsAAACxsgAAABArCwAAALGyAAAAECsLAAAAsbIAAAAQKwsAAACxsgAAABArCwAAALGyAAAAECsLAAAAsbIAAAAQKwsAAACxsgAAABArCwAAALGyAAAAECsLAAAAsbIAAAAQKwsAAACxsgAAABArCwAAALGyAAAAECsLAAAAsbIAAAAQKwsAAACxsgAAABArCwAAALGyAAAAECsLAAAAsbIAAAAQKwsAAACxsgAAABArCwAAALGyAAAAECsLAAAAsbIAAAAQKwsAAACxspAT7733nt59913VdtZZZwkAAOz6LOTE9OnTdc0116i2EIJSqZQAAMCuzQIAAECsLAAAAMTKAgAAQKwsAAAAxMoCAABArCwAAADEygIAAECsLAAAAMTKAgAAQKwsAAAAxMoCAABArCwAAADEygIAAECsLAAAAMTKAgAAQKwsAAAAxMoCAABArCwAAADEygIAAECsLAAAAMTKAgAAQKwsAAAAxMoCAABArCwAAADEygIAAECsLAAAAMTKAgAAQKwsAAAAxMoCAABArCwAAADEygIAAECsLAAAAMTKAgAAQKwsAAAAxMoCAABArCwAAADEygIAAECsLAAAAMTKAgAAQKwsAAAAxMoCAABArCwAAADEysoTzzzzjNatW6ctFRQUqKSkROXl5SopKdGWFi9erLffflvfpHPnziotLRUAAMD3ZeWJwYMH65///Ke2paCgQOedd57uuusuNWrUSDWeeuopjRgxQt/kscce07nnnisAAIDvy8oj6XRa06dPV20hBK1YsUL333+/pk2bpvXr1+u//uu/tKUhQ4aoZ8+e2pajjjpKAAAAcbDySGFhoc466yxtzYABA1ReXq4nnnhCn332mRo1aqTa2rdvr7POOksAAAC5Zu0i9tlnH/3oRz/S3/72Ny1evFidO3fWt/Wvf/1Ly5YtU9OmTdW0aVPV9v7772v16tVq06aNGjRoIAAAgG2xdiFr165VZN9999V3kUqldPbZZ6uoqEhvv/22iouLFfnXv/6lo446Svvvv79ee+01AQAAbI+1i3jxxRc1f/58HXvssSovL9eWQgjKZDLamj322EOFhYVq1KiR7r//fp155pkaOXKk7rrrLkWGDh2qNWvWaObMmapTp44AAAC2x8oj2WxW5513nmqrrq7WBx98oL///e/q1q2bKisrtTWXX365Lr/8cm3Nww8/rL59+ypyxhlnaMCAAbrnnnvUq1cvrV69WjNmzNDNN9+s9u3ba0el02nVrVtXmzdvVjabVeSYY44RgGR59913VVhYqBrz589Xp06dBCCZpk6dqlatWinprDyyadMmPfHEE6pt8+bN2rhxoyIlJSUqKirS1nTp0kWHHHKItqZVq1aq7Y477tALL7ygiy++WGvWrNFxxx2n4cOH69sYOXKkRo4cqXfeeUfDhw8XgGTq0qWLAOSPVCqlfGDlkTp16iiTyWhLn3zyiUaNGqU//elPaty4sW6//XZtqW/fvhoyZIh2xF577aUHH3xQ3bt3V1FRkSZPnqzCwkJ9F+3atdOzzz4rAACw+7B2AY0bN9bdd9+tadOm6fnnn1ccVq5cqcj69etVVVWlZs2aCQAAYEdYu4i6deuqadOm+uSTT/R9rVixQldccYW6d++u9957TwMHDtTChQu11157CQAA4JtYu5i1a9fq+9i0aZP69eungoICTZw4UQsXLtRpp52mq666ShMmTBAAAMA3sXYhRUVFWr9+vb744gs1bNhQ38Vtt92ml156SQ888ICaNm2qpk2b6qKLLtIDDzygc889Vz179hQAAMD2WLuQQw45RG+99ZauvvpqjR8/Xt/WwoUL9Zvf/EY9evTQwIEDVeP222/XX//6Vw0aNEiLFi1SgwYNBAAAsC1WnnjssceUzWa1PTfffLMuueQSFRcXq0afPn30ox/9SAcffLC+Sd26dfXcc8+poqJCte2zzz6aM2eOVq5cqUwmowYNGggAAGBbrDzRuXNnfZMDDzxQBx54oGorKytTWVmZdkR5ebnKy8u1Na1atVKrVq0EAADwTSzk3JtvvqlHH31UAJKpU6dOOvvsszVt2jQtWLBAAJKpX79+atOmjfKBhZxbsGCBbr75ZgFIpkGDBunss8/WE088oenTpwtAMnXq1Elt2rRRPrCQc9lsVpGVK1eqtLRUAJKjefPmqq6uViSbzap9+/Z66623BCA53nnnHVVUVCiEoHxhIedCCIrYFoBksa0QgiIhBNkWgGSxrUgIQfnCQs6FEBSxLQDJYlshBEVCCLItAMliW5EQgvKFhZwLIShiWwCSxbZCCIqEEGRbAJLFtiIhBOULCzkXQlDEtgAki22FEBQJIci2ACSLbUVCCMoXFnIuhKCIbQFIFtsKISgSQtAee+whAMliW5EQgvKFhZwLIShiWwCSxbZCCIqEEFRUVCQAyWJbkRCC8oWFnKuurlZBQYFSqZQAJItthRAUCSHItgAki21FQgjKFxZyLoQg2wKQPLYVQlAkhCDbApAsthUJIShfWMi5EIJsC0Dy2NbGjRsVCSHItgAki21FQgjKFxZyLoQg2wKQPLa1bt06RUIIsi0AyWJbkRCC8oWFnAshyLYAJI9thRAUCSHItgAkSyqVUkFBgaqrq5UvLORcCEG2BSB5bCuEoEgIQbYFIHlsK4SgfGEh50IIsi0AyWNbIQRFQgiyLQDJY1shBOULCzkXQpBtAUge2wohKBJCkG0BSB7bCiEoX1jIuRCCbAtA8thWCEGREIJsC0Dy2FYIQfnCQs6FEGRbAJLHtkIIioQQZFsAkse2QgjKFxZyLoQg2wKQPLaVzWYVCSHItgAkj21ls1nlCws5F0KQbQFIHtsKISgSQpBtAUge2wohKF9YyLkQgmwLQPLYVghBkRCCbAtA8thWCEH5wkLOhRBkWwCSJ51OK4SgSAhBtgUgedLptEIIyhcWci6EINsCkDy2FUJQdXW1Nm/erFQqJQDJY1shBOULCzmXzWZlWwCSx7ZCCAohKGJbAJLHtkIIyhcWci6EINsCkDy2FUJQNptVxLYAJI9thRCULyzkXAhBderUEYDkSaVSCiEom80qkk6nBSB5bCuEoHxhIedCCEqn0wKQPLYV2bBhgyK2BSB5bCuEoHxhIedCCLItAMljW5FMJqOIbQFIHtsKIShfWMi5EIJsC0Dy2FYkk8koYlsAkse

> **Gradient orientation syntax**
>
> For people familiar with python, the syntax
> `gradient=(orientation=[1.0, 0.0, 0.0],)` might be confusing.
> `(orientation=[1.0, 0.0, 0.0],)` creates a [named
> tuple](https://docs.julialang.org/en/v1/manual/types/#Named-Tuple-Types-1)
> with one element. You can think of this as like a python dictionary
> that cannot be changed after creation. In `MRIBuilder` these named
> tuples are used to specify the parameters of individual sequence
> blocks, such as the diffusion gradient.

MCMRSimulator can also take Pulseq `.seq` files directly as input. We
use MRIBuilder here only to create a quick example; for more realistic
simulations, we recommend using sequences generated with Pulseq.
MRIBuilder has been discontinued. In the future, we expect to recommend
[`decseq`](https://git.fmrib.ox.ac.uk/ndcn0236/decseq/) for quick
experiments where a fully accurate sequence is not required like here,
but it is not yet in a usable state.

Use
[`MRIBuilder.adjust`](https://open.oxcin.ox.ac.uk/pages/ndcn0236/mribuilder.jl/stable/adjust_sequences/)
to create four versions of this sequence with b-values of 0, 1, 2, and 3
ms/μm². The b-value scales with the square of the gradient strength, so
the gradient scale is the square root of the desired b-value relative to
the maximum. The sequences keep the same timing and gradient
orientation.

In [1]:
bvals = [0, 1, 2, 3]
gradient_scales = sqrt.(bvals ./ maximum(bvals))
sequences = MRIBuilder.adjust(
    base_sequence,
    diffusion=(scale=gradient_scales,),
    merge=false,
)
length(sequences)

4

> **Julia array operations**
>
> In `python` most functions such as `a + b` work both on scalars or
> numpy arrays. In `julia`, such functions only work on scalars. To
> apply a function elementwise to an array, use the dot syntax:
> `f.(array)` or `a .+ b`. For example, `sqrt.(bvals)` applies the
> square root to each element of `bvals`.

## 3. Set biophysical parameters and simulate

The
[`Simulation`](https://pages.fmrib.ox.ac.uk/ndcn0236/mcmrsimulator.jl/dev/api/#MCMRSimulator.Simulations.Simulation)
combines the sequences and geometry with the tissue parameters. We keep
diffusivity fixed at `2 μm²/ms`, `R1` at `1/1000 ms⁻¹`, and `R2` at
`1/80 ms⁻¹` while comparing the different b-values. Relaxation rates are
specified in `ms⁻¹`.

In [1]:
simulation = Simulation(
    sequences;
    geometry=geometry,
    diffusivity=2.0,
    R1=1 / 1000,
    R2=1 / 80,
    verbose=false,
)

Simulation(fixed(Rotate{Repeat{GeometryVector{Shift{Round{2}}}}}), D=2.0um^2/ms, GlobalProperties(R1=0.001kHz, R2=0.0125kHz, )):
4 sequences:
Sequence(
  scanner=MRIBuilder.Scanners.Scanner(1.5, 0.0017030553904000003, 0.007237985409200001),
  delay=0.0,
  echo_time=68.23394213515668,
  readout_times=[68.23394213515668],
  repetition_time=68.23394213556276,
  diffusion_time=34.11697114828762,
  TE=68.23394213515668,
  Δ=34.11697114828762,
  TR=68.23394213556276,
  duration=68.23394213556276,
  excitation=BuildingBlock(
    duration=0.0,
    t=0.0: InstantPulse(
      flip_angle=90.0,
      phase=0.0,
      effective_time=0.0,
      inverse_bandwidth=0.0,
      duration=0.0,
    ),
  ),
  gradient=Trapezoid1D(
    rise_time=0.2352951746295513,
    flat_time=33.6463806364067,
    slew_rate=0.0,
    orientation=[1.0, 0.0, 0.0],
    group=:diffusion,
    readout_times=Any[],
    δ=33.881675811036246,
    gradient_strength=[0.0, 0.0, 0.0],
    duration=34.1169709856658,
    qvec=[0.0, 0.0, 0

This `Simulation` object fully defines the experiment, including the
geometry, sequences, and biophysical parameters. We can now run the
simulation to get the resulting signals using the
[`readout`](https://pages.fmrib.ox.ac.uk/ndcn0236/mcmrsimulator.jl/dev/api/#MCMRSimulator.Evolve.readout)
function. This function determines how many spins to simulate and can be
used to override things like when to readout the signal. Rather than
explicitly define the number of spins, we can use the `target_snr`
argument to specify the desired signal-to-noise ratio (SNR) for the
worst-case signal. The simulator will run until it reaches that SNR,
which is useful for Monte Carlo simulations where the required number of
spins needed can vary depending on the geometry and sequence.

In [1]:
signals = readout(simulation; target_snr=3)

4-element Vector{MCMRSimulator.Spins.SpinOrientationSum}:
 SpinOrientationSum(longitudinal=1.1250355529972975, transverse=426.1668781431999, phase=90.0°, nspins=1000, snr=[Inf, 6.092924823026576e15, 8.683397981858495e15])
 SpinOrientationSum(longitudinal=1.1250355529973521, transverse=347.32850392345705, phase=89.97879337308707°, nspins=1000, snr=[71.02318536720583, 57.00301595370983, 8.699915111634191e15])
 SpinOrientationSum(longitudinal=1.1250355529973506, transverse=321.70467931742195, phase=88.4509697726542°, nspins=1000, snr=[64.7272828783158, 43.98253307649156, 8.693924207316932e15])
 SpinOrientationSum(longitudinal=1.1250355529973581, transverse=315.2871601371969, phase=87.9168783321539°, nspins=1000, snr=[61.869854053006584, 42.01008186762262, 8.688526347581321e15])

The output is a four-element vector, with one result for each sequence
in the same order as `bvals`: 0, 1, 2, and 3. Each `SpinOrientationSum`
reports the magnetisation summed across the spins in that readout:

- `transverse` is the magnitude of the **net** transverse magnetisation.
  As b-value increases, spins accumulate more varied phases and cancel
  more when combined, so this signal decreases. The geometry influences
  how spins move while the diffusion gradients are applied.
- `longitudinal` is the net magnetisation along z. These values are also
  sums; use `longitudinal(signal; mean=true)` or
  `transverse(signal; mean=true)` to get the per-spin average.
- `phase` is the angle of the net transverse signal. A phase near 90°
  means the net signal points mostly along the positive y direction.
- `nspins` is the number of spins included in the sum.
- `snr` gives estimated SNRs for the x, y, and z magnetisation
  components. Very large values or `Inf` indicate effectively zero
  estimated uncertainty for a component, not an infinite physical
  signal. These infinite SNRs occur when each spin gives the exact same
  contribution to the net signal, which can happen if the random motion
  of the spins does not affect the signal evolution (as is the case for
  all longitudinal signals).

At b=0, the diffusion gradients are scaled to zero, but spins still
diffuse and experience relaxation. Thus, b=0 is the reference signal,
not a simulation with diffusion switched off.

Note that this workflow is quite different from some other Monte Carlo
simulators. In these, your simulator might simply give you a trajectory
for each spins, which can then be used to calculate the signal for any
sequence post-hoc. In `MCMRSimulator`, the sequence or sequences are
part of the simulation and the simulator calculates the signal for those
sequences directly. This approach allows the simulator to: 1. use the
sequence to dynamically adjust the timestep of the simulator (e.g.,
reduce the timestep during an RF pulse) leading to faster results; 2.
model the effect of collisions on the spin magnetisation, which is
crucial for the accurate simulation of magnetisation transfer and
surface relaxation effects. It is possible to get the spin trajectories
from a simulation using the `return_snapshot=true` option in
[`readout`](https://pages.fmrib.ox.ac.uk/ndcn0236/mcmrsimulator.jl/dev/api/#MCMRSimulator.Evolve.readout),
but this is not the default behaviour.

## 4. Explore the effect of diffusion weighting

We can extract the transverse signal at the spin echo from each readout.
The `transverse` function returns the transverse signal at the spin echo
for each sequence. The `mean=true` option averages over all spins in the
simulation. We then normalize the signals by dividing by the first
signal, which corresponds to b=0. Finally, we plot this transverse
signal attenuation against the b-value.

In [1]:
transverse_signals = transverse.(signals; mean=true)
normalized_signals = transverse_signals ./ first(transverse_signals)
fig = Figure(size=(700, 400))
ax = Axis(
    fig[1, 1];
    xlabel="b-value",
    ylabel="Transverse signal attenuation",
)
scatterlines!(ax, bvals, normalized_signals, markersize=12)
lines!(ax, bvals, b -> exp(-b * 2.) * first(normalized_signals), color=:black)
fig

For comparison, the black line shows the expected signal decay for free
diffusion with the same diffusivity. The simulated signal decays more
slowly because the cylinders restrict diffusion perpendicular to their
axes.

In most experiments you probably just want to extract the
[`transverse`](https://pages.fmrib.ox.ac.uk/ndcn0236/mcmrsimulator.jl/dev/api/#MCMRSimulator.Spins.transverse)
and
[`phase`](https://pages.fmrib.ox.ac.uk/ndcn0236/mcmrsimulator.jl/dev/api/#MCMRSimulator.Spins.phase)
aspects of the signal, which are the parts of the signal that are
actually measured in an MRI experiment. However, you can of course also
access other aspects of the spin magnetisation using the functions
listed in the [API
docs](https://pages.fmrib.ox.ac.uk/ndcn0236/mcmrsimulator.jl/dev/api/#Querying-simulation-output).
Note that some of these will require a `return_snapshot=true` option in
the `readout` function to return the full spin state at the end of the
simulation, rather than just the summed magnetisation.

### Compare intracellular and extracellular signals

We can use
[`Subset`](https://pages.fmrib.ox.ac.uk/ndcn0236/mcmrsimulator.jl/dev/api/#MCMRSimulator.Subsets.Subset)
to read out spins inside the cylinders separately from spins outside
them. Previously, our results were just a vector with one element per
sequence. Now we specify a vector of subsets, so the output is a matrix
with one row per sequence and one column per subset. In general,
`readout` can return up to a 4-dimensional array with each dimension
corresponding to sequences, subsets, multiple repetition times, and
multiple readouts within a single repetition.

In [1]:
compartment_signals = readout(
    simulation;
    target_snr=3,
    subset=[Subset(inside=true), Subset(inside=false)],
)

intracellular = compartment_signals[:, 1]
extracellular = compartment_signals[:, 2]
intracellular_signal = transverse.(intracellular; mean=true)
extracellular_signal = transverse.(extracellular; mean=true)

intracellular_attenuation = intracellular_signal ./ first(intracellular_signal)
extracellular_attenuation = extracellular_signal ./ first(extracellular_signal)

compartment_figure = Figure(size=(850, 400))
intracellular_axis = Axis(
    compartment_figure[1, 1];
    title="Intracellular",
    xlabel="b-value",
    ylabel="Normalized transverse signal",
)
extracellular_axis = Axis(
    compartment_figure[1, 2];
    title="Extracellular",
    xlabel="b-value",
)
linkyaxes!(intracellular_axis, extracellular_axis)
scatterlines!(intracellular_axis, bvals, intracellular_attenuation)
scatterlines!(extracellular_axis, bvals, extracellular_attenuation)
compartment_figure

Each compartment is normalized to its own b=0 signal. This compares the
diffusion-related attenuation per spin without conflating it with the
different numbers of spins inside and outside the cylinders.

In this example, there is only attenuation in the extracellular
compartment, because the diffusion gradients are applied perpendicular
to the cylinder axes. Spins inside the cylinders cannot move enough in
that direction to accumulate sufficient phase to cause signal
attenuation.

Monte Carlo readouts have sampling noise, so small differences between
points are not necessarily meaningful. Try changing the geometry radius,
fixed diffusivity, b-value range, or relaxation rates, then compare how
the signal changes. How would you change the acquisition to get signal
attenuation even in the intracellular compartment? In the next workshop
sections we will extend these building blocks to more complex
morphologies, magnetisation transfer, and susceptibility-driven
dephasing.